In [ ]:
import time
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords

# Download NLTK stopwords dataset
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import pandas as pd
from google.colab import files

# 1. Run this first to open the upload button
uploaded = files.upload()

# 2. Load the uploaded files into pandas
df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

# 3. Label and combine
df_fake['label'] = 1
df_true['label'] = 0
df = pd.concat([df_fake, df_true], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
df['content'] = df['title'].fillna('') + " " + df['text'].fillna('')

Saving Fake.csv to Fake (2).csv


### 1.3 Data Cleaning & Text Normalization

In [ ]:
import os

# Ensure correct file loading regardless of Colab renaming duplicate uploads
fake_file = 'Fake (1).csv' if os.path.exists('Fake (1).csv') else 'Fake.csv'
true_file = 'True (1).csv' if os.path.exists('True (1).csv') else 'True.csv'

df_fake = pd.read_csv(fake_file)
df_true = pd.read_csv(true_file)

# Labeling and merging
df_fake['label'] = 1  # Fake news
df_true['label'] = 0  # Real news

df = pd.concat([df_fake, df_true], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
df['content'] = df['title'].fillna('') + " " + df['text'].fillna('')

# 1. Drop missing values and duplicates
df = df.dropna(subset=['content']).drop_duplicates(subset=['content']).reset_index(drop=True)

# 2. Define preprocessing function
def preprocess_text(text):
    text = text.lower()                                                # Lowercase normalization
    text = re.sub(r'https?://\S+|www\.\S+', '', text)                  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)                            # Remove special chars and digits
    tokens = [word for word in text.split() if word not in stop_words] # Remove stopwords
    return " ".join(tokens)

# Apply preprocessing
df['clean_content'] = df['content'].apply(preprocess_text)
print(f"Data Cleaning Completed! Total Cleaned Articles: {len(df)}")

Data Cleaning Completed! Total Cleaned Articles: 39105


## Module 2: Feature Extraction & Baseline Modeling
### 2.1 Dataset Splitting

In [ ]:
from sklearn.model_selection import train_test_split

# Define feature variable (X) and target label (y)
X = df['clean_content']
y = df['label']

# Perform 80/20 train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples (80%): {len(X_train)}")
print(f"Testing samples  (20%): {len(X_test)}")

Training samples (80%): 31284
Testing samples  (20%): 7821


### 2.2 Text Vectorization (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer with a maximum limit of 5,000 features
tfidf = TfidfVectorizer(max_features=5000)

# Fit vectorizer on training data and transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF Training Matrix Shape: {X_train_tfidf.shape}")
print(f"TF-IDF Testing Matrix Shape:  {X_test_tfidf.shape}")

TF-IDF Training Matrix Shape: (31284, 5000)
TF-IDF Testing Matrix Shape:  (7821, 5000)


### 2.3 Baseline Model Selection & Experimental Evaluation

In [ ]:
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define candidate baseline models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

# Train and evaluate candidate algorithms
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)

    # Measure inference speed on test dataset
    start_time = time.time()
    y_pred = model.predict(X_test_tfidf)
    end_time = time.time()

    inference_speed_ms = ((end_time - start_time) / len(X_test)) * 1000

    # Store performance evaluation metrics
    results.append({
        "Algorithm Name": name,
        "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (Test)": round(precision_score(y_test, y_pred), 4),
        "Recall (Test)": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "Inference Speed (ms)": round(inference_speed_ms, 4)
    })

# Display filled metrics table
metrics_df = pd.DataFrame(results)
print("=== Baseline Model Comparative Performance ===")
print(metrics_df.to_string(index=False))

=== Baseline Model Comparative Performance ===
         Algorithm Name  Accuracy (%)  Precision (Test)  Recall (Test)  F1-Score  Inference Speed (ms)
    Logistic Regression         98.93            0.9932         0.9832    0.9882                0.0014
Multinomial Naive Bayes         93.65            0.9295         0.9319    0.9307                0.0037
          Decision Tree         99.51            0.9930         0.9964    0.9947                0.0012
          Random Forest         99.81            0.9983         0.9975    0.9979                0.0557
